# CPU : adopter batch001, puis publier les lots pilotes
Notebook de procédure, sans résultats préremplis. Première session : `PHASE='adopt'`.
Conserver le JSON d'adoption hors de la VM. Ouvrir ensuite un **nouveau notebook CPU**,
réutiliser ce notebook avec `PHASE='verify_and_plan'` et coller ce JSON.
Le dataset manuel n'est ni recréé ni retéléchargé. Ne copier aucun marqueur de session.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
CODE_SOURCE = Path('/kaggle/input/datasets/nnnnbhss/geofirex-code-v3/GeoFire-X-2026')  # adapter au dataset de code v3
CODE = Path('/kaggle/working/GeoFire-X-2026-v3')
assert CODE_SOURCE.is_dir(), 'Renseigner le chemin du code v3 attaché au notebook'
if not CODE.exists():
    shutil.copytree(CODE_SOURCE, CODE)
manifest = json.loads((CODE_SOURCE/'code_manifest.json').read_text())
assert all(hashlib.sha256((CODE/name).read_bytes()).hexdigest() == digest for name, digest in manifest.items()), 'Code local différent du paquet v3'
os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kagglehub>=1.0.3,<2'], check=True)
sys.path.insert(0, str(CODE/'src'))
from kagglehub.auth import get_username
assert get_username() == 'nnnnbhss', 'Compte Kaggle authentifié inattendu'
print('Compte vérifié :', get_username())


In [ ]:
PHASE = 'adopt'  # puis 'verify_and_plan' dans un NOUVEAU notebook CPU
MANUAL_ROOT = Path('/kaggle/input/datasets/nnnnbhss/geofire-x-wsts-batch001')
MANUAL_HANDLE = 'nnnnbhss/geofire-x-wsts-batch001'
INDEX_PATH = Path('data/manifests/real/corpus_zip_index_v1.json')
INDEX = json.loads(INDEX_PATH.read_text())
ADOPTION_PROOF_JSON = ''  # coller le JSON de la première session, pas un reçu d'upload inventé
def save_once(path, value):
    from geofirex.corpus import json_new
    if path.exists():
        assert json.loads(path.read_text()) == value, 'Choisir une nouvelle sortie pour un résultat différent'
    else:
        json_new(path, value)


## Session A — adoption en lecture seule


In [ ]:
if PHASE == 'adopt':
    from geofirex.kaggle_persistence import adopt_manual, session_id
    proof = adopt_manual(INDEX, MANUAL_ROOT, MANUAL_HANDLE, session=session_id())
    save_once(Path('/kaggle/working/manual-adoption.json'), proof)
    print(json.dumps(proof, indent=2))
    print('Conserver ce JSON hors de la VM, puis ouvrir un nouveau notebook CPU.')


## Session B — vérification indépendante et certificat bootstrap


In [ ]:
if PHASE == 'verify_and_plan':
    from geofirex.kaggle_persistence import verify_fresh_session, session_id, validate_bootstrap
    assert ADOPTION_PROOF_JSON.strip(), 'Coller le JSON d’adoption provenant de la session A'
    proof = json.loads(ADOPTION_PROOF_JSON)
    certificate = verify_fresh_session(proof, session=session_id())
    validate_bootstrap(certificate, INDEX)
    BOOTSTRAP = Path('/kaggle/working/bootstrap-manual-verified.json')
    save_once(BOOTSTRAP, certificate)
    print(json.dumps(certificate, indent=2))
    print('Conserver également ce certificat hors de la VM.')


## Tester l'API automatique avant plusieurs Gio
Cette sonde publie 60 minuscules fichiers texte (aucune donnée scientifique), teste aussi le ZIP automatique et relit tous les hashes. Elle ne remplace pas le certificat du corpus manuel. Activer seulement cette petite publication pour valider les permissions de création réelles avant les gros lots.


In [ ]:
RUN_PUBLICATION_PROBE = False
if PHASE == 'verify_and_plan' and RUN_PUBLICATION_PROBE:
    import uuid
    subprocess.run([sys.executable, '-m', 'scripts.kaggle_publication_probe',
        '--job', '/kaggle/working/publication-probe-v3', '--handle', 'nnnnbhss/geofirex-publication-probe-v3',
        '--out', f'/kaggle/working/publication-probe-{uuid.uuid4().hex[:8]}.json', '--execute'], check=True)


## Quotas réels et plans sans transfert
Renseigner le stockage **privé encore disponible** dans le compte Kaggle. Les 3 Gio de transfert par lot sont un plafond à approuver, pas un quota supposé. Le disque est mesuré et le ZIP d'upload est budgété.


In [ ]:
PRIVATE_FREE_GIB = None  # OBLIGATOIRE : valeur vérifiée dans votre compte Kaggle
PLAN_DIR = Path('/kaggle/working/pilot-v3')
if PHASE == 'verify_and_plan':
    from scripts.prepare_kaggle_pilot import prepare
    from geofirex.cloud_corpus import validate_plan
    assert PRIVATE_FREE_GIB is not None and PRIVATE_FREE_GIB > 0, 'Renseigner le quota privé disponible'
    private_bytes = int(PRIVATE_FREE_GIB*1024**3)
    free = shutil.disk_usage('/kaggle/working').free
    allocation = min(free, 8*1024**3)
    if not PLAN_DIR.exists():
        report = prepare(INDEX, PLAN_DIR, private_free_bytes=private_bytes,
                         storage_bytes=allocation, roots=[MANUAL_ROOT])
    else:
        report = json.loads((PLAN_DIR/'pilot.json').read_text())
        assert report['private_quota_declared_bytes'] == private_bytes, 'Nouveaux quotas : choisir un autre PLAN_DIR'
        for label in ['train2018', 'val2020', 'extension2019']:
            validate_plan(json.loads((PLAN_DIR/f'{label}.json').read_text()), INDEX)
    print(json.dumps(report, indent=2))


## Commandes exactes — affichées avant exécution


In [ ]:
PUBLICATION_MODE = 'create'  # 'resume' seulement pour vérifier une publication possiblement déjà réussie
def acquisition_command(label):
    plan = json.loads((PLAN_DIR/f'{label}.json').read_text())
    limits = plan['limits']
    minimum, maximum = (30, 40) if label == 'train2018' else (20, 30)
    args = [sys.executable, '-m', 'scripts.kaggle_acquire_publish',
        '--index', str(INDEX_PATH), '--job', f'/kaggle/working/jobs-v3/{label}',
        '--handle', f'nnnnbhss/geofire-x-wsts-{label}-pilot01', '--years', plan['events'][0][:4],
        '--min-events', str(minimum), '--max-events', str(maximum), '--plan', str(PLAN_DIR/f'{label}.json'),
        '--bootstrap-verification', str(BOOTSTRAP), '--acquired-roots', str(MANUAL_ROOT),
        '--publication-mode', PUBLICATION_MODE]
    for key, flag in [('transfer_bytes','transfer'), ('storage_bytes','storage'),
                      ('upload_bytes','upload'), ('private_free_bytes','private-free')]:
        args += [f'--{flag}-bytes', str(limits[key])]
    return args
if PHASE == 'verify_and_plan':
    import shlex
    for label in ['train2018', 'val2020', 'extension2019']:
        print(label, ':', shlex.join(acquisition_command(label)+['--execute']))


## Exécution autorisée — une cellule par lot
Passer uniquement le booléen du lot approuvé à `True`. La cellule publie et relit la version privée avant de se terminer. Conserver le reçu affiché hors de la VM. Après une interruption d'acquisition, relancer le même job ; après une réponse d'upload perdue, utiliser `PUBLICATION_MODE='resume'`. Aucun fichier local n'est supprimé.


In [ ]:
RUN_TRAIN_2018 = False
if PHASE == 'verify_and_plan' and RUN_TRAIN_2018:
    subprocess.run(acquisition_command('train2018')+['--execute'], check=True)


In [ ]:
RUN_VAL_2020 = False
if PHASE == 'verify_and_plan' and RUN_VAL_2020:
    subprocess.run(acquisition_command('val2020')+['--execute'], check=True)


In [ ]:
RUN_EXTENSION_2019 = False
if PHASE == 'verify_and_plan' and RUN_EXTENSION_2019:
    subprocess.run(acquisition_command('extension2019')+['--execute'], check=True)


## Récupérer les reçus pour le notebook GPU indépendant
Ces petits JSON doivent être conservés hors de `/kaggle/working`. Le dataset privé contient les TIFF ; les reçus servent à vérifier les versions et la nouvelle session.


In [ ]:
if PHASE == 'verify_and_plan':
    receipts = []
    for label in ['train2018', 'val2020', 'extension2019']:
        folder = Path('/kaggle/working/jobs-v3')/label
        for path in sorted(folder.glob('publication-*.json')):
            receipt = json.loads(path.read_text())
            if receipt['status'] == 'REMOTE_VERIFIED_FRESH_SESSION_PENDING':
                receipts.append(receipt)
    print(json.dumps(receipts, indent=2))
